# E-commerce Sales Analytics – Data Cleaning

Pandas data-cleaning notebook for the E-commerce Sales Analytics project.

In [1]:
import pandas as pd
from pathlib import Path

## 1. File paths

In [2]:
input_path = Path("../data/ecommerce_sales_raw.csv")
output_path = Path("../data/ecommerce_sales_cleaned.csv")

## 2. Load raw dataset

In [3]:
df = pd.read_csv(input_path)

print("Raw dataset shape:", df.shape)
df.head()

Raw dataset shape: (5000, 12)


,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
0,10001,01-01-2022,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20
1,10002,01-02-2022,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10
2,10003,01-03-2022,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35
3,10004,01-04-2022,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90
4,10005,01-05-2022,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56


## 3. Standardize column names

In [4]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("Columns after standardization:")
print(df.columns.tolist())

Columns after standardization:
['order_id', 'order_date', 'customer_id', 'product_category', 'region', 'quantity', 'unit_price', 'discount', 'payment_method', 'delivery_days', 'customer_rating', 'revenue']


## 4. Remove duplicate rows

In [5]:
before_duplicates = len(df)
df = df.drop_duplicates().copy()
print("Rows removed:", before_duplicates - len(df))
print("Shape after removing duplicates:", df.shape)

Rows removed: 0
Shape after removing duplicates: (5000, 12)


## 5. Clean text columns

In [6]:
text_columns = [
    "product_category",
    "region",
    "payment_method"
]

for col in text_columns:
    df[col] = df[col].astype("string").str.strip()

print("Text columns cleaned:", text_columns)

Text columns cleaned: ['product_category', 'region', 'payment_method']


## 6. Clean order date

In [7]:
df["order_date"] = pd.to_datetime(
    df["order_date"].astype("string").str.strip(),
    format="mixed",
    errors="coerce"
)

print("Date conversion completed.")

Date conversion completed.


## 7. Convert numeric columns

In [8]:
numeric_columns = [
    "order_id", "customer_id", "quantity", "unit_price",
    "discount", "delivery_days", "customer_rating", "revenue"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)

order_id                     int64
order_date          datetime64[ns]
customer_id                  int64
product_category    string[python]
region              string[python]
quantity                     int64
unit_price                 float64
discount                   float64
payment_method      string[python]
delivery_days                int64
customer_rating            float64
revenue                    float64
dtype: object


## 8. Handle missing / invalid values

In [9]:
required_columns = [
    "order_id", "order_date", "customer_id", "product_category",
    "region", "quantity", "unit_price", "discount",
    "payment_method", "delivery_days", "customer_rating", "revenue"
]

missing_before = df[required_columns].isna().sum().sum()
df = df.dropna(subset=required_columns).copy()

print("Missing/invalid required values removed:", missing_before)
print("Shape after handling missing values:", df.shape)

Missing/invalid required values removed: 0
Shape after handling missing values: (5000, 12)


## 9. Business-rule validation

In [10]:
before_rules = len(df)

df = df[
    (df["quantity"] > 0) &
    (df["unit_price"] >= 0) &
    (df["discount"].between(0, 1)) &
    (df["delivery_days"] >= 0) &
    (df["customer_rating"].between(1, 5)) &
    (df["revenue"] >= 0)
].copy()

print("Rows removed by business rules:", before_rules - len(df))
print("Shape after business-rule validation:", df.shape)

Rows removed by business rules: 0
Shape after business-rule validation: (5000, 12)


## 10. Validate revenue

In [11]:
calculated_revenue = (
    df["quantity"] * df["unit_price"] * (1 - df["discount"])
)

revenue_difference = (
    df["revenue"] - calculated_revenue
).abs()

print("Maximum revenue difference:", revenue_difference.max())
assert revenue_difference.max() <= 0.01
print("Revenue validation passed.")

Maximum revenue difference: 0.005000000000109139
Revenue validation passed.


## 11. Round numeric fields

In [12]:
df["unit_price"] = df["unit_price"].round(2)
df["discount"] = df["discount"].round(2)
df["revenue"] = df["revenue"].round(2)
df["customer_rating"] = df["customer_rating"].round(1)

print("Numeric fields rounded.")

Numeric fields rounded.


## 12. Standardize date format

In [13]:
df["order_date"] = df["order_date"].dt.strftime("%Y-%m-%d")
print("Date format:", df["order_date"].head().tolist())

Date format: ['2022-01-01', '2022-01-02', '2022-01-03', '2022-01-04', '2022-01-05']


## 13. Sort dataset

In [14]:
df = df.sort_values("order_id").reset_index(drop=True)
print("Dataset sorted by order_id.")

Dataset sorted by order_id.


## 14. Final validation

In [15]:
assert df["order_id"].is_unique
assert df.isna().sum().sum() == 0

print("Final validation passed.")
print("Final dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Missing values:", df.isna().sum().sum())

Final validation passed.
Final dataset shape: (5000, 12)
Duplicate rows: 0
Missing values: 0


## 15. Export cleaned dataset

In [16]:
df.to_csv(output_path, index=False)

print("Cleaning completed successfully.")
print("Final dataset shape:", df.shape)
print("Output file:", output_path)

Cleaning completed successfully.
Final dataset shape: (5000, 12)
Output file: /mnt/data/ecommerce_sales_cleaned.csv


## Final cleaned dataset preview

In [17]:
df.head(10)

,order_id,order_date,customer_id,product_category,region,quantity,unit_price,discount,payment_method,delivery_days,customer_rating,revenue
0,10001,2022-01-01,1102,Beauty,South,7,373.65,0.28,Wallet,10,4.7,1883.20
1,10002,2022-01-02,1435,Clothing,South,7,47.74,0.09,Card,6,3.9,304.10
2,10003,2022-01-03,1860,Beauty,East,3,311.28,0.31,COD,6,2.5,644.35
3,10004,2022-01-04,1270,Electronics,West,5,524.47,0.02,Wallet,6,1.6,2569.90
4,10005,2022-01-05,1106,Clothing,West,5,139.87,0.33,Wallet,4,4.9,468.56
5,10006,2022-01-06,1071,Electronics,West,2,264.44,0.20,Card,3,3.3,423.10
6,10007,2022-01-07,1700,Clothing,South,6,564.60,0.23,Card,1,1.8,2608.45
7,10008,2022-01-08,1020,Beauty,North,6,265.71,0.13,Card,2,3.7,1387.01
8,10009,2022-01-09,1614,Clothing,South,6,224.65,0.27,Card,2,1.4,983.97
9,10010,2022-01-10,1121,Clothing,West,4,272.71,0.06,Card,10,4.5,1025.39
